In [ ]:
"""
Created on  Apr 26 
@author: Jingyi
"""
"""
Created on Tues Mar 29 16:12:01 2022
@author: Oumbeg
"""
#------------------------------------------------ Import Lib ----------------------------------------
import re
from bs4 import BeautifulSoup
import os
import pandas as pd
from time import sleep
import datetime
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import pdfplumber
    
    

In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'ZA SARB' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.2.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running ZA SARB Web Scraping Tool v.2.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



In [4]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}   

   
regdict={
        'ZA SARB 1': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/sa-registered-banks-and-representative-offices', 
         'ZA SARB 2': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/sa-registered-banks-and-representative-offices', 
         'ZA SARB 3': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/sa-registered-banks-and-representative-offices', 
         'ZA SARB 4': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/sa-registered-banks-and-representative-offices', 
         'ZA SARB 5': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/sa-registered-banks-and-representative-offices',
         'ZA SARB 6': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/sa-registered-banks-and-representative-offices',
        'ZA SARB 7': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/SA_registered_financial_institutions',
        'ZA SARB 8': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/SA_registered_financial_institutions',
         'ZA SARB 9': 'https://www.resbank.co.za/en/home/what-we-do/Prudentialregulation/SA_registered_financial_institutions'
        }



Typology={

        regulatorName+' 1': 'Branches of Foreign Banks',
        regulatorName+' 2': 'Foreign Bank Representatives',
        regulatorName+' 3': 'Foreign Controlled Banks',
        regulatorName+' 4': 'Locally Controlled Banks',
        regulatorName+' 5': 'Mutual Banks',
        regulatorName+' 6': 'Banks in Liquidation',
        regulatorName+' 7': 'Licensed Insurers',
        regulatorName+' 8': 'Registered Co-operative Banks',
        regulatorName+' 9': 'Registered Co-operative financial institution',


        }
processdate = now.strftime('%Y-%m-%d')

pattern = re.compile('^[0-9]*$')
my_dict = {'Banks in Liquidation': '6', 'Branches of Foreign Banks': '1', 'Foreign Bank Representatives': '2',
          'Foreign Controlled Banks': '3', 'Locally Controlled Banks': '4', 'Mutual Banks':'5'}
cumul_prev_len_as2 = 0


print('The current folder is: {}\nThe temp folder is: {}'.format(scriptfolder, tempfolder))

The current folder is: C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\ZA SARB
The temp folder is: C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\ZA SARB\tempfolder


In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------
def findEmail(myData):
    """
    This function finds email in a string.
    :param myData: string
    :return: String
    """
    regex = re.compile('[a-zA-Z0-9.\-_]+@[^@]+\.[^@]+')
    email = regex.findall(myData)
    return ' '.join([str(elem) for elem in email])

def findUrl(mydata):
    regex = re.compile(r"(?i)\b((?:https?://|www\d{2,4,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:'\".,<>?«»“”‘’]))")
    url = regex.findall(mydata)
    return ' '.join([str(elem) for elem in [x[0] for x in url]])



def findZip(mydata):
    regex = re.compile(r'\b\d{4}\b')
    zip_codes = regex.findall(mydata)
    return zip_codes[-1] if zip_codes else ''

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict

<>:10: SyntaxWarning: invalid escape sequence '\-'
<>:10: SyntaxWarning: invalid escape sequence '\-'
C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_25736\1333459980.py:10: SyntaxWarning: invalid escape sequence '\-'
  regex = re.compile('[a-zA-Z0-9.\-_]+@[^@]+\.[^@]+')


In [ ]:
# %%


#------------------------------------------------ Begin_ fileName ----------------------------------------

for reg in regdict:
    
    print('Working with {}'.format(reg))
    driver.get(regdict[reg])
    sleep(5)
    

    # the block where to find the links 
    soup = BeautifulSoup(driver.page_source, "html.parser")

    # Download the file
    
    listname_website2 = 'SA '+ Typology[reg].lower()
    
    try:
        element1 = driver.find_element(By.LINK_TEXT, Typology[reg])
        driver.execute_script("arguments[0].click();", element1)
    except:
        sleep(3)
        element2 = driver.find_element(By.LINK_TEXT, listname_website2)
        driver.execute_script("arguments[0].click();", element2)
    sleep(3)
    
    dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
    tables = []
    if  reg!='ZA SARB 7':
        print(os.listdir(tempfolder))
        with pdfplumber.open(dl_files[0]) as pdf:
            for page in pdf.pages:
                table = page.extract_table()
                #print(table)
                tables.append(table)
    if reg != 'ZA SARB 2' and reg!='ZA SARB 6' and reg!='ZA SARB 7' and reg!='ZA SARB 8' and reg!='ZA SARB 9':
        for t in range(len(tables)):
            if t == 0:
                # Column name
                Column_name = tables[t][0]
                #print(Column_name)
                content = tables[t][1:]

                for c in range(len(content)):
                    if content[c][0]:
                        name = content[c][0].replace('\n',' ')
                        address = content[c][1].replace('\n',' ')
                        tel = content[c][3].replace('\n',' ')
                        website = content[c][4].replace('\n',' ')
                        #print(name,address,tel,website)
                        sqldict['Name'].append(name)
                        sqldict['Address_1'].append(address)
                        sqldict['Zip'].append(findZip(address))
                        sqldict['Phone'].append(tel)
                        sqldict['Website'].append(website)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['RegCtry'].append(reg.split()[0])
                        sqldict['RegCode'].append(reg.split()[1])
                        sqldict['ListCode'].append(reg.split()[2])
                        sqldict['ListName'].append(Typology[reg])
                sqldict = bourange_same_length_array(sqldict)


            else:
                    contents = tables[t][:]
                    for c in range(len(contents)):
                        name = contents[c][0].replace('\n',' ')
                        address = contents[c][1].replace('\n',' ')
                        tel = contents[c][3].replace('\n',' ')
                        website = contents[c][4].replace('\n',' ')
                        #print(name,address,tel,website)
                        sqldict['Name'].append(name)
                        sqldict['Address_1'].append(address)
                        sqldict['Zip'].append(findZip(address))
                        sqldict['Phone'].append(tel)
                        sqldict['Website'].append(website)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['RegCtry'].append(reg.split()[0])
                        sqldict['RegCode'].append(reg.split()[1])
                        sqldict['ListCode'].append(reg.split()[2])
                        sqldict['ListName'].append(Typology[reg])
                    sqldict = bourange_same_length_array(sqldict)
    elif reg == 'ZA SARB 2':
            for t in range(len(tables)):
                if t == 0:
                    # Column name
                    Column_name = tables[t][0]
                    #print(Column_name)
                    content = tables[t][1:]
                    for c in range(len(content)):
                        name = content[c][0].replace('\n',' ')
                        address = content[c][-1].replace('\n',' ')
                        tel = content[c][2].replace('\n',' ')
                        website = content[c][3].replace('\n',' ')
                        #print(name,address,tel,website)
                        sqldict['Name'].append(name)
                        sqldict['Address_1'].append(address)
                        sqldict['Zip'].append(findZip(address).split(' ')[-1])
                        sqldict['Phone'].append(tel)
                        sqldict['Website'].append(website)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['RegCtry'].append(reg.split()[0])
                        sqldict['RegCode'].append(reg.split()[1])
                        sqldict['ListCode'].append(reg.split()[2])
                        sqldict['ListName'].append(Typology[reg])
                    sqldict = bourange_same_length_array(sqldict)
                else:
                    contents = tables[t][:]
                    for c in range(len(contents)):
                        
                        if contents[c][0] == 'Name' or contents[c][0] is None:
                            #print('Header & Wrong')
                            pass
                        else:

                            name = contents[c][0].replace('\n',' ')
                            address = contents[c][-1].replace('\n',' ')
                            tel = contents[c][2].replace('\n',' ')
                            website = contents[c][3].replace('\n',' ')
                            #print(name,address,tel,website)  
                            sqldict['Name'].append(name)
                            sqldict['Address_1'].append(address)
                            sqldict['Zip'].append(findZip(address).split(' ')[-1])
                            sqldict['Phone'].append(tel)
                            sqldict['Website'].append(website)
                            sqldict['ListProcessDate'].append(processdate)
                            sqldict['RegulationType'].append('Regulated')
                            sqldict['RegCtry'].append(reg.split()[0])
                            sqldict['RegCode'].append(reg.split()[1])
                            sqldict['ListCode'].append(reg.split()[2])
                            sqldict['ListName'].append(Typology[reg])
                        sqldict = bourange_same_length_array(sqldict)

    elif reg == 'ZA SARB 6':
            for t in range(len(tables)):
                if t == 0:
                    # Column name
                    Column_name = tables[t][0]
                    #print(Column_name)
                    content = tables[t][1:]
                    for c in range(len(content)):
                        if 'Name' in content[c][0] :
                            continue
                        else:
                            name = content[c][0].replace('\n',' ')
                            address = content[c][1].replace('\n',' ')
                            zip_code = content[c][2].replace('\n',' ')
                            phone_  = content[c][4].replace('\n',' ')
                            tel = content[c][5].replace('\n',' ')
                            website =content[c][-1].replace('\n',' ')

                        sqldict['Name'].append(name)
                        sqldict['Address_1'].append(address)
                        sqldict['Zip'].append(zip_code)
                        sqldict['Phone'].append(tel)
                        sqldict['Website'].append(website)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['RegCtry'].append(reg.split()[0])
                        sqldict['RegCode'].append(reg.split()[1])
                        sqldict['ListCode'].append(reg.split()[2])
                        sqldict['ListName'].append(Typology[reg])

                    sqldict = bourange_same_length_array(sqldict)

    elif reg == 'ZA SARB 7':
        sleep(3)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        sleep(3)
        data = soup.find('table',class_ = 'table').find_all('tr')
        for i in data:
            tds = i.find_all('td')
            if len(tds)>0:
                name = tds[0].text
                id = tds[-1].text
                #print(name,id)
                sqldict['Name'].append(name)
                sqldict['InternalID_1_type'].append('Insurer Number')
                sqldict['InternalID_1'].append(id)
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegulationType'].append('Regulated')
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
            sqldict = bourange_same_length_array(sqldict)
    elif reg == 'ZA SARB 8':
        begin = 0
        infos = tables[-1]
        for index,info in enumerate(infos):
            if info[0] == 'Name':
                begin = index + 1
        #print(begin)
        new_infos = infos[begin:]
        for index,info in enumerate(new_infos):
            name = info[0].replace('\n',' ')
            regis_num = info[1].replace('\n',' ')
            address = info[2].replace('\n',' ')
            zip = findZip(address)
            contact_details = info[3]
            email = contact_details.split('\n')[-1]
            sqldict['Name'].append(name)
            sqldict['Address_1'].append(address)
            sqldict['Zip'].append(zip.split(' ')[-1])
            sqldict['Email'].append(email)
            sqldict['InternalID_1_type'].append('Registration Number as a Co-operative')
            sqldict['InternalID_1'].append(regis_num)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
            sqldict = bourange_same_length_array(sqldict)
    elif reg == 'ZA SARB 9':
        for t in range(len(tables)):
            if t == 0:
                # Column name
                Column_name = tables[t][0]
                #print(Column_name)
                content = tables[t][1:]
                for c in range(len(content)):
                    name = content[c][0].replace('\n',' ')
                    date = content[c][1].replace('\n',' ')
                    address = content[c][-1].replace('\n',' ')
                    zip = findZip(address)
                    #print(name,date,address)
                    sqldict['Name'].append(name)
                    sqldict['RegulationDate'].append(date)
                    sqldict['Address_1'].append(address)
                    sqldict['Zip'].append(zip.split(' ')[-1])
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])
                sqldict = bourange_same_length_array(sqldict)
            else:
                contents = tables[t][:]
                for c in range(len(contents)):
                    name = contents[c][0].replace('\n',' ')
                    date = contents[c][1].replace('\n',' ')
                    address = contents[c][-1].replace('\n',' ')
                    zip = findZip(address)
                    #print(name,date,address)
                    sqldict['Name'].append(name)
                    sqldict['RegulationDate'].append(date)
                    sqldict['Address_1'].append(address)
                    sqldict['Zip'].append(zip.split(' ')[-1])
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])
                sqldict = bourange_same_length_array(sqldict)
            
    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))
                        

                        
    

    

Working with ZA SARB 1
['Branches Foreign_July 2025.pdf']
Working with ZA SARB 2
['Foreign Banks Representative_ 2 Sept 2025.pdf']
Working with ZA SARB 3
['Foreign Controlled 1.pdf']
Working with ZA SARB 4
['Locally Controlled Banks as at 10 Oct 2025.pdf']
Working with ZA SARB 5
['Mutual Banks as at July 2025.pdf']
Working with ZA SARB 6
['Banks in Liquidation Report 1.pdf']
Working with ZA SARB 7
[]
Working with ZA SARB 8
['Register of Co-ops Banks-Membership and deposits_November 2024.pdf']
Working with ZA SARB 9
["PA registered CFI's as at December 2024.pdf"]


In [7]:
# check which is different

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 260 values.
Key 'priority' has 260 values.
Key 'ListLabel' has 260 values.
Key 'Typology' has 260 values.
Key 'EntryType' has 260 values.
Key 'Name' has 260 values.
Key 'InternalID_1' has 260 values.
Key 'InternalID_1_type' has 260 values.
Key 'InternalID_2' has 260 values.
Key 'InternalID_2_type' has 260 values.
Key 'InternalID_3' has 260 values.
Key 'InternalID_3_type' has 260 values.
Key 'CoType' has 260 values.
Key 'License_Type' has 260 values.
Key 'Address_1' has 260 values.
Key 'Address_2' has 260 values.
Key 'City' has 260 values.
Key 'Zip' has 260 values.
Key 'Cntry' has 260 values.
Key 'Phone' has 260 values.
Key 'Fax' has 260 values.
Key 'Website' has 260 values.
Key 'Email' has 260 values.
Key 'RegulationType' has 260 values.
Key 'RegulationTypeCode' has 260 values.
Key 'RegulationDate' has 260 values.
Key 'CancellationDate' has 260 values.
Key 'RegCtry' has 260 values.
Key 'RegCode' has 260 values.
Key 'ListCode' has 260 values.
Key 'ListLanguage' has 260 v

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
1,,,,,,Islamic Bank Limited (In Final Liquidation),,,,,...,,,,,,,,,,
2,,,,,,Habib Overseas Bank Limited (under curatorship),,,,,...,,,,,,,,,,
3,,,,,,Regal Treasury Private Bank Limited (In liquid...,,,,,...,,,,,,,,,,
4,,,,,,VBS Mutual Bank,,,,,...,,,,,,,,,,


In [8]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df = df[df['Name'] != 'Name']
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()
sleep(3)

driver.quit()

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_25736\4058894465.py:8: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [9]:

filename = '{} SQL Ready {}.csv'.format(regulatorName, str(now).replace(":",".")[:-7])
df.to_csv(filename)